In [13]:
import glob
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

import numpy as np

def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    ds_djf['time'] = pd.date_range(str(ds_djf.time[0].values).split('-')[0], str(int(str(ds_djf.time[-1].values).split('-')[0]) +1), freq='YS-FEB',unit='s')
    return ds_djf


def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')


def load_climatology(path):
    clim = xr.open_dataset(glob.glob(path)[0])
    clim['time'] = [1,2,3,4,5,6,7,8,9,10,11,12]
    clim = clim.rename({'time': 'month'})
    return clim

def load_data(path):
    ens = []
    ds = []
    for p in glob.glob(path):
        ens.append(p.split('/')[-1].split('_')[4])
        ds.append(xr.open_dataset(p))
    data = xr.concat(ds, dim='ens')
    data['ens'] = ens

    return data

def pressure_weighting(data, plevs=slice(7000,2000)):
    # here I select the pressure level intervall 
    # for which I want to caluculate the layer weights
    # (in this case plevs = 70 - 20 hPa)
    data = data.sel(plev=plevs)
    # Then natural logarithm of this pressure values is caluculated
    log_pressure = np.log(data.plev.values)  
    # the difference between this logaritmic pressure values is made
    log_layer_thickness = np.abs(np.diff(log_pressure))
    # then i devide these differences by the total thickness of the layer
    # = sum of the differences
    log_layer_thickness = log_layer_thickness / sum(log_layer_thickness)

    # this part just creates an array to make it compatible with the data 
    # #(no manipulation of the results done in this step)
    log_layer_thickness = xr.DataArray(
        log_layer_thickness,
        coords={"plev": data.plev.values[:-1]},  # Assign to the lower pressure level of each layer
        dims=["plev"]
    )

    # finally I multiply the data with the "normalized" thickness of the pressure layer
    data = data * log_layer_thickness

    return data.sum(dim='plev')

In [14]:
path = "/sto4/data/Intermediate/VolMIP/pinatubo-full/"
path_piControl = "/sto4/data/Intermediate/VolMIP/piControl/"
models = ['CanESM5', 'GISS-E2-1-G','MPI-ESM1-2-LR', 'MIROC-ES2L', 'UKESM1-0-LL', 'IPSL-CM6A-LR']

nao_flags = pd.read_csv('/home/kperny/VolMIP_stratosphere/tab/nao_flags.csv')
nao_flags = nao_flags.set_index(['model', 'realization'])


In [15]:
# calculation for piControl
for model in models:
    # open climatology
    ua_clim_pi = load_climatology(path_piControl + model + '/ua_clim_AmonZ*')
    ta_clim_pi = load_climatology(path_piControl + model + '/ta_clim_AmonZ*')
    
    # open temperature and wind data
    ta_tmp = xr.open_dataset(glob.glob(path_piControl + model + '/ta_AmonZ*.nc')[0], decode_times=time_coder)
    ta_ano = anomaly_calculation(ta_tmp, ta_clim_pi)
    ta_piControl = calculate_djf_mean(ta_ano)

    ua_tmp = xr.open_dataset(glob.glob(path_piControl + model + '/ua_AmonZ*.nc')[0], decode_times=time_coder)
    ua_ano = anomaly_calculation(ua_tmp, ua_clim_pi)
    ua_piControl = calculate_djf_mean(ua_ano)

    # aggregate tropical (-20,20) and polar (70,90) temperature between 70-20hPa and 10 hPa wind data (60,70) for pinatubo-full years
    nao_flags.loc[model,'deltaT_tropics'] = pressure_weighting(ta_piControl.sel(time = nao_flags.loc[model]['year'].values, method='nearest').ta.sel(lat=slice(-20,20)).mean('lat')).squeeze().values
    nao_flags.loc[model,'deltaT_polar'] = pressure_weighting(ta_piControl.sel(time = nao_flags.loc[model]['year'].values, method='nearest').ta.sel(lat=slice(70,90)).mean('lat')).squeeze().values
    nao_flags.loc[model,'deltaU'] = ua_piControl.sel(time = nao_flags.loc[model]['year'].values, method='nearest').ua.sel(lat=slice(60,70), plev=1000).mean('lat').squeeze().values

nao_flags.reset_index().to_csv('/home/kperny/VolMIP_stratosphere/tab/ta_ua_piControl.csv', index=False)
nao_flags.reset_index().to_csv('/home/kperny/VolMIP_stratosphere/tab/diagnostics_piControl.csv', index=False)

/tmp/ipykernel_2980724/1070168184.py:27: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(glob.glob(path)[0])
/tmp/ipykernel_2980724/1070168184.py:27: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(glob.glob(path)[0])
/tmp/ipykernel_2980724/1070168184.py:22: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ano_list.ap

In [16]:
nao_flags

year    nao_pi  flag_pi  nao_volc  flag_volc  \
model       realization                                                       
CanESM5     r1i1p2f1     5552-02-01 -0.001352        0  0.619078          1   
            r2i1p2f1     5557-02-01  0.125872        0 -1.742433         -1   
            r3i1p2f1     5575-02-01  1.143758        1  0.715891          1   
            r4i1p2f1     5622-02-01  0.211339        0  0.104238          0   
            r5i1p2f1     5643-02-01 -0.593414       -1 -1.025887         -1   
...                             ...       ...      ...       ...        ...   
UKESM1-0-LL r21i1p1f2    3123-02-01  2.542346        1  1.365885          1   
            r22i1p1f2    3074-02-01  1.863322        1 -0.023668          0   
            r23i1p1f2    3104-02-01  1.020190        1  1.619856          1   
            r24i1p1f2    3077-02-01  0.504877        1 -1.474555         -1   
            r25i1p1f2    3181-02-01  1.424958        1  0.389597          0   

                         deltaT_tropics  deltaT_polar     deltaU  
model       realization                                           
CanESM5     r1i1p2f1           0.375639     -3.202295   8.716141  
            r2i1p2f1           0.343064     -5.434440  10.561366  
            r3i1p2f1           0.607024     -3.045932   5.480005  
            r4i1p2f1           0.476886     -4.454523   9.615352  
            r5i1p2f1           0.094850      1.320145  -2.981189  
...                                 ...           ...        ...  
UKESM1-0-LL r21i1p1f2          1.423468     -2.407708   4.661552  
            r22i1p1f2          0.765512     -3.358174   6.989256  
            r23i1p1f2          0.566508      4.815586  -8.566979  
            r24i1p1f2          0.027857      6.289649 -10.239064  
            r25i1p1f2          0.637946     -0.332594   0.424457  

[261 rows x 8 columns]

In [8]:
enso_nao_flags

nao      enso  nao_flag  enso_flag        year  \
model       realization                                                        
CanESM5     r1i1p2f1    -0.000646 -0.827322         0         -1  5552-02-01   
            r2i1p2f1     0.125736  0.130379         0          0  5557-02-01   
            r3i1p2f1     1.144113 -0.765208         1         -1  5575-02-01   
            r4i1p2f1     0.211332 -0.375304         0          0  5622-02-01   
            r5i1p2f1    -0.593341 -0.363156        -1          0  5643-02-01   
...                           ...       ...       ...        ...         ...   
UKESM1-0-LL r21i1p1f2    2.542339 -1.069389         1         -1  3123-02-01   
            r22i1p1f2    1.863309  0.208591         1          0  3074-02-01   
            r23i1p1f2    1.020208  0.018721         1          0  3104-02-01   
            r24i1p1f2    0.504888  0.283152         1          0  3077-02-01   
            r25i1p1f2    1.424974  0.793467         1          1  3181-02-01   

                         deltaT_tropics  deltaT_polar     deltaU  
model       realization                                           
CanESM5     r1i1p2f1           0.375639     -3.202295   8.716141  
            r2i1p2f1           0.343064     -5.434440  10.561366  
            r3i1p2f1           0.607024     -3.045932   5.480005  
            r4i1p2f1           0.476886     -4.454523   9.615352  
            r5i1p2f1           0.094850      1.320145  -2.981189  
...                                 ...           ...        ...  
UKESM1-0-LL r21i1p1f2               NaN           NaN        NaN  
            r22i1p1f2               NaN           NaN        NaN  
            r23i1p1f2               NaN           NaN        NaN  
            r24i1p1f2               NaN           NaN        NaN  
            r25i1p1f2               NaN           NaN        NaN  

[236 rows x 8 columns]